<a href="https://colab.research.google.com/github/SeoYoung346/2025.LLM/blob/main/Claude%EC%99%80_%ED%95%A8%EA%BB%98%ED%95%98%EB%8A%94_%EB%8F%84%EA%B5%AC_%EC%82%AC%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 설치

In [1]:
!pip install -q anthropic

2. 클라이언트와 헬퍼 함수

In [2]:
import json
import anthropic
from anthropic.types import Message, ToolParam
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
model = "claude-sonnet-5-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[], tools=None, tool_choice=None, thinking=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    if tool_choice:
        params["tool_choice"] = tool_choice
    if thinking:
        params["thinking"] = thinking

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

3. 도구 함수

In [3]:
from datetime import datetime, timedelta
import calendar


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + (month - 1) // 12
        month = (month - 1) % 12 + 1
        day = min(date.day, calendar.monthrange(year, month)[1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_year = date.year + duration
        if date.month == 2 and date.day == 29 and not calendar.isleap(new_year):
            new_date = date.replace(year=new_year, day=28)
        else:
            new_date = date.replace(year=new_year)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

In [4]:
# 함수 단독 테스트
print(get_current_datetime())
print(add_duration_to_datetime("2025-01-31", 1, "months"))
set_reminder("치과 예약", "2025-02-28 10:00:00")

2026-11-03 21:42:09
Friday, February 28, 2025 12:00:00 AM
----
Setting the following reminder for 2025-02-28 10:00:00:
치과 예약
----


4. 도구 스키마

In [5]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time formatted according to the specified format string. "
        "Use this tool when the user asks about the current date, time, or both. "
        "The format follows Python's strftime conventions."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A string specifying the format of the returned datetime, using Python strftime codes. "
                    "Examples: '%Y-%m-%d' for 2024-01-31, '%H:%M:%S' for 14:30:45."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a specified duration to a datetime string and returns the resulting datetime "
        "in a detailed format (e.g. 'Thursday, April 03, 2025 10:30:00 AM'). "
        "Handles seconds, minutes, hours, days, weeks, months, and years, including month-end "
        "and leap-year edge cases. Use this tool when you need to calculate a future or past date."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string, in the format given by input_format.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add. Use a negative value to subtract.",
                "default": 0,
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration.",
                "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
                "default": "days",
            },
            "input_format": {
                "type": "string",
                "description": "The strftime format of datetime_str, e.g. '%Y-%m-%d' or '%Y-%m-%d %H:%M:%S'.",
                "default": "%Y-%m-%d",
            },
        },
        "required": ["datetime_str"],
    },
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": (
        "Creates a timed reminder that will notify the user at the specified time with the "
        "provided content. Use this tool when the user wants to be reminded of something "
        "at a specific date and time."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be shown in the reminder notification.",
            },
            "timestamp": {
                "type": "string",
                "description": "When to trigger the reminder, in ISO 8601 format (YYYY-MM-DDTHH:MM:SS).",
            },
        },
        "required": ["content", "timestamp"],
    },
})

5. 메시지 블록 처리

In [6]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = chat(messages, tools=[get_current_datetime_schema])

for block in response.content:
    print(block.type, "→", block)
print("stop_reason:", response.stop_reason)

tool_use → ToolUseBlock(id='toolu_01Hk5pjcpvovnRW4YtPooFwg', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)
stop_reason: tool_use


6. 도구 결과 전송

In [7]:
add_assistant_message(messages, response)

tool_use = next(block for block in response.content if block.type == "tool_use")
result = get_current_datetime(**tool_use.input)

add_user_message(messages, [
    {
        "type": "tool_result",
        "tool_use_id": tool_use.id,
        "content": result,
        "is_error": False,
    }
])

final_response = chat(messages, tools=[get_current_datetime_schema])
print(text_from_message(final_response))

The exact time is **21:42:27**. That's the time when I checked, so it's a few seconds old by now.


7. 멀티턴 대화와 도구

In [8]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            content = tool_output if isinstance(tool_output, str) else json.dumps(tool_output, ensure_ascii=False)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": content,
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages, tools):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

8. 여러 도구 사용하기

In [9]:
reminder_tools = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
]

messages = []
add_user_message(messages, "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050.")
run_conversation(messages, tools=reminder_tools)



----
Setting the following reminder for 2050-06-27T09:00:00:
Doctor's appointment today
----
I set a reminder for your doctor's appointment on Monday, June 27, 2050. That date is 177 days after January 1, 2050.

You didn't give a time, so I set it for 9:00 AM. The reminder tool returned an empty response instead of a confirmation, so I can't be sure it saved. If you tell me the real appointment time, I can set a new reminder for that time.


[{'role': 'user',
  'content': "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01TMFi374iJLpdpA1Rh6Niky', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TMFi374iJLpdpA1Rh6Niky',
    'content': 'Monday, June 27, 2050 12:00:00 AM',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='CAQSoQcKEAgTGAI4AUIIdGhpbmtpbmcSDB1CLTxHWl+p422++hoMr8VWPm/ERad1rWHcIjC1+EF0XkpO4puSupUBy3qv5qEsWazB/Tlxzz0AfIIy3aZJ7WwTVBOTdX6BjeBMYZIqvgbOcsusrkhDkTf9Q2TxiipM25qUwqsECjApptdAiLwEJNDsfc5GYG/G2daeE2XAStkl0Rsmeun5kBNTr3PPTcCzPg1xl0wAqpxmiZUWt0Wo5uh9VjMoPe1ORX6XhP2NOr5jX6aniXV5qZT1Nl750UkQP7F8mYSDWELaE5DuS6nbzdSL1pMYv/ZH/XU75oU7ydVAt+syCOU96k/

9. 배치 도구

In [10]:
batch_tool_schema = ToolParam({
    "name": "batch_tool",
    "description": (
        "Invoke multiple other tool calls simultaneously. "
        "Use this when several independent tool calls are needed at the same time."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "The name of the tool to invoke"},
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
})


def run_batch(invocations=[]):
    batch_output = []
    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        tool_output = run_tool(name, args)
        batch_output.append({"tool_name": name, "output": tool_output})
    return batch_output


# run_tool에 batch_tool 분기 추가
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [11]:
messages = []
add_user_message(
    messages,
    "Set two reminders for Jan 1, 2050 at 8am: one for 'Go to the gym' and one for 'Take vitamins'.",
)
run_conversation(messages, tools=reminder_tools + [batch_tool_schema])


----
Setting the following reminder for 2050-01-01T08:00:00:
Go to the gym
----
----
Setting the following reminder for 2050-01-01T08:00:00:
Take vitamins
----
I set both reminders for January 1, 2050 at 8:00 AM:
1. "Go to the gym"
2. "Take vitamins"

Each call returned `null` instead of a confirmation message. That may be normal for this tool, but I can't confirm the reminders were saved. It would be worth checking your reminders list.


[{'role': 'user',
  'content': "Set two reminders for Jan 1, 2050 at 8am: one for 'Go to the gym' and one for 'Take vitamins'."},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_014pLLu6njFwuFVZmvGucXyR', caller=DirectCaller(type='direct'), input={'content': 'Go to the gym', 'timestamp': '2050-01-01T08:00:00'}, name='set_reminder', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01Njccb7U8FFf54f8RHZUzte', caller=DirectCaller(type='direct'), input={'content': 'Take vitamins', 'timestamp': '2050-01-01T08:00:00'}, name='set_reminder', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_014pLLu6njFwuFVZmvGucXyR',
    'content': 'null',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01Njccb7U8FFf54f8RHZUzte',
    'content': 'null',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='I set both reminders for January 1, 2050 a

10. 구조화된 데이터를 위한 도구

In [15]:
article_summary_schema = ToolParam({
    "name": "article_summary",
    "description": "Creates a summary of an article with its key insights.",
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "The title of the article"},
            "author": {"type": "string", "description": "The name of the author"},
            "key_insights": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of 3-5 key insights from the article",
            },
        },
        "required": ["title", "author", "key_insights"],
    },
})

article = """
Title: The Quiet Power of Sleep
By Jane Park

New research shows that consistent sleep schedules matter as much as total sleep time.
Participants who went to bed at the same time each night reported better focus and mood,
even when they slept slightly less. Researchers also found that screen use in the hour
before bed delayed sleep onset by an average of 20 minutes. The authors recommend a fixed
bedtime, a dark room, and avoiding screens late at night.
"""

messages = []
add_user_message(
    messages,
    f"Summarize this article using the article_summary tool:\n<article>\n{article}\n</article>",
)

response = chat(
    messages,
    tools=[article_summary_schema],
    tool_choice={"type": "auto"},
    system="You must always respond by calling the article_summary tool. Do not answer in plain text.",
)

tool_use = next((block for block in response.content if block.type == "tool_use"), None)
if tool_use:
    print(json.dumps(tool_use.input, ensure_ascii=False, indent=2))
else:
    print("도구를 호출하지 않았어요. 텍스트 응답:\n", text_from_message(response))

{
  "title": "The Quiet Power of Sleep",
  "author": "Jane Park",
  "key_insights": [
    "Consistent sleep schedules matter as much as total sleep time.",
    "Participants who went to bed at the same time each night reported better focus and mood, even when they slept slightly less.",
    "Screen use in the hour before bed delayed sleep onset by an average of 20 minutes.",
    "The authors recommend a fixed bedtime, a dark room, and avoiding screens late at night."
  ]
}


Part 2. 텍스트 편집 도구

11. 텍스트 편집 도구 구현

In [16]:
import os
import shutil


class TextEditorTool:
    def __init__(self, base_dir="", backup_dir=""):
        self.base_dir = os.path.abspath(base_dir or os.getcwd())
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path):
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(f"Access denied: path '{file_path}' is outside the allowed directory")
        return abs_path

    def _backup_file(self, file_path):
        if os.path.exists(file_path):
            backup_path = os.path.join(self.backup_dir, os.path.basename(file_path) + ".bak")
            shutil.copy2(file_path, backup_path)

    def view(self, file_path, view_range=None):
        path = self._validate_path(file_path)

        if os.path.isdir(path):
            return "\n".join(sorted(os.listdir(path)))
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(path, encoding="utf-8") as f:
            lines = f.readlines()

        start, end = 1, len(lines)
        if view_range:
            start = max(1, view_range[0])
            end = len(lines) if view_range[1] == -1 else min(len(lines), view_range[1])

        return "".join(f"{i}: {lines[i - 1]}" for i in range(start, end + 1))

    def str_replace(self, file_path, old_str, new_str):
        path = self._validate_path(file_path)
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(path, encoding="utf-8") as f:
            content = f.read()

        count = content.count(old_str)
        if count == 0:
            raise ValueError("No match found for replacement text")
        if count > 1:
            raise ValueError(f"Found {count} matches. Provide more context for a unique match.")

        self._backup_file(path)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content.replace(old_str, new_str))
        return "Successfully replaced text at exactly one location."

    def create(self, file_path, file_text):
        path = self._validate_path(file_path)
        if os.path.exists(path):
            raise FileExistsError(f"File already exists: {file_path}")

        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(file_text)
        return f"Successfully created {file_path}"

    def insert(self, file_path, insert_line, new_str):
        path = self._validate_path(file_path)
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(path, encoding="utf-8") as f:
            lines = f.readlines()
        if insert_line < 0 or insert_line > len(lines):
            raise ValueError(f"insert_line {insert_line} is out of range (0-{len(lines)})")

        self._backup_file(path)
        if not new_str.endswith("\n"):
            new_str += "\n"
        lines.insert(insert_line, new_str)
        with open(path, "w", encoding="utf-8") as f:
            f.writelines(lines)
        return f"Successfully inserted text after line {insert_line}"

12. 스키마와 run_tool 연결

In [17]:
# 직접 작성하지 않고, Anthropic이 정의한 타입과 이름만 지정
text_editor_schema = {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
}

text_editor_tool = TextEditorTool()


def run_text_editor(tool_input):
    command = tool_input["command"]
    path = tool_input["path"]

    if command == "view":
        return text_editor_tool.view(path, tool_input.get("view_range"))
    elif command == "str_replace":
        return text_editor_tool.str_replace(path, tool_input["old_str"], tool_input["new_str"])
    elif command == "create":
        return text_editor_tool.create(path, tool_input["file_text"])
    elif command == "insert":
        new_str = tool_input.get("insert_text", tool_input.get("new_str", ""))
        return text_editor_tool.insert(path, tool_input["insert_line"], new_str)
    else:
        raise ValueError(f"Unknown text editor command: {command}")


# run_tool에 텍스트 편집 도구 분기 추가
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
    elif tool_name == "str_replace_based_edit_tool":
        return run_text_editor(tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

13. 파일 읽고 요약하기

In [18]:
with open("main.py", "w", encoding="utf-8") as f:
    f.write('''def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b
''')

messages = []
add_user_message(messages, "Open the ./main.py file and summarize its contents.")
run_conversation(messages, tools=[text_editor_schema])


`./main.py` is a short file with two functions:

- **`greet(name)`** takes a name and returns the string `"Hello, {name}!"`.
- **`add(a, b)`** takes two values and returns their sum (`a + b`).

The file has no imports, no entry point (`if __name__ == "__main__":`) and no other logic. It looks like a small utility module that other code would import.


[{'role': 'user',
  'content': 'Open the ./main.py file and summarize its contents.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01XHCQkPM81s7iEKkGXauLE3', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './main.py'}, name='str_replace_based_edit_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01XHCQkPM81s7iEKkGXauLE3',
    'content': '1: def greet(name):\n2:     return f"Hello, {name}!"\n3: \n4: \n5: def add(a, b):\n6:     return a + b\n',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='`./main.py` is a short file with two functions:\n\n- **`greet(name)`** takes a name and returns the string `"Hello, {name}!"`.\n- **`add(a, b)`** takes two values and returns their sum (`a + b`).\n\nThe file has no imports, no entry point (`if __name__ == "__main__":`) and no other logic. It looks like a small utility module that other co

14. 코드 수정하고 테스트 파일 만들기

In [19]:
messages = []
add_user_message(
    messages,
    "Open ./main.py and add a function that calculates pi to the 5th decimal place. "
    "Then create a test.py file that tests your implementation.",
)
run_conversation(messages, tools=[text_editor_schema])



I added `calculate_pi()` to `main.py` and wrote tests for it in `test.py`. I haven't run the tests because I can't execute code here. To run them, use `python test.py` or `python -m unittest test`.

**`calculate_pi(decimals=5)` in `main.py`**
- It uses Machin's formula, `π = 16·arctan(1/5) − 4·arctan(1/239)`, with integer arithmetic. This avoids floating-point error and doesn't depend on `math.pi`.
- It computes 10 extra digits and then rounds, so the last decimal place comes out correct.
- With the default it returns `3.14159`. The `decimals` argument lets you ask for other precisions.

**`test.py`**
The tests use `unittest` and cover:
- the default result, `3.14159`
- agreement with `round(math.pi, 5)`
- closeness to `math.pi` within 5 places
- other precisions (2 and 10 decimals)
- that the return type is `float`


[{'role': 'user',
  'content': 'Open ./main.py and add a function that calculates pi to the 5th decimal place. Then create a test.py file that tests your implementation.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_0121yYcXwMBxSuGduk6k4AVP', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './main.py'}, name='str_replace_based_edit_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_0121yYcXwMBxSuGduk6k4AVP',
    'content': '1: def greet(name):\n2:     return f"Hello, {name}!"\n3: \n4: \n5: def add(a, b):\n6:     return a + b\n',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='CAQSkwYKEAgTGAI4AUIIdGhpbmtpbmcSDOL5CN78zRe5ivhbBxoMRLO+MiouLXGbQ5+IIjAgKw5tWoJj2bjrjGbKsjOogO8Tq349gOnGp/CDk07vvQ5ze9Ow7hNuJZrWDLFHZYAqsAVQ1NtINBUmFIvTnCp1kBXiQ+KIzdjArpFfRaoOBfkZNTfSWu7SDO1Karm9c0v0JqY6apo3rTOfwoOHW2lGDwUB2kcMevX5Mny/rKq9jSsKPYcnsb2CN2pvkW6TC

In [20]:
# 결과 확인
print(open("main.py", encoding="utf-8").read())
print("=" * 40)
print(open("test.py", encoding="utf-8").read())

def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b



def calculate_pi(decimals=5):
    """Calculate pi rounded to the given number of decimal places (default 5).

    Uses Machin's formula, pi = 16*arctan(1/5) - 4*arctan(1/239),
    with integer arithmetic and guard digits so rounding is accurate.
    """
    guard = 10
    scale = 10 ** (decimals + guard)

    def arctan_inv(x):
        # arctan(1/x) as a scaled integer, via the Taylor series
        total = term = scale // x
        x_squared = x * x
        n = 1
        sign = 1
        while term:
            term //= x_squared
            n += 2
            sign = -sign
            total += sign * (term // n)
        return total

    pi_scaled = 16 * arctan_inv(5) - 4 * arctan_inv(239)
    # Drop the guard digits, rounding to nearest
    pi_rounded = (pi_scaled + 10 ** guard // 2) // 10 ** guard
    return pi_rounded / 10 ** decimals

import math
import unittest

from main import calculate_pi


clas

In [21]:
!python test.py

.....
----------------------------------------------------------------------
Ran 5 tests in 0.001s

OK


Part 3. 웹 검색 도구

15. 기본 웹 검색

In [22]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

messages = []
add_user_message(messages, "What's the best exercise for gaining leg muscle?")
response = chat(messages, tools=[web_search_schema])

for block in response.content:
    print(block.type)
print("stop_reason:", response.stop_reason)

text
stop_reason: end_turn


16. 응답 블록 살펴보기 (검색어, 검색 결과, 인용)

In [23]:
for block in response.content:
    if block.type == "server_tool_use":
        print(f"[검색어] {block.input['query']}")
    elif block.type == "web_search_tool_result":
        if isinstance(block.content, list):
            for r in block.content:
                print(f"  - {r.title}\n    {r.url}")
        else:
            print(f"  검색 오류: {block.content.error_code}")
    elif block.type == "text":
        print(block.text, end="")
        for c in block.citations or []:
            print(f"\n    ↳ 출처: {c.title} ({c.url})", end="")
        print()

The squat is generally considered the best single exercise for building leg muscle. It works the quads, glutes, and adductors hard, and it also involves the hamstrings and core. No single move covers everything, though, so the best approach is a few key lifts.

**Core lifts**
- **Squats** (back, front, or goblet) are the foundation for quads and glutes.
- **Romanian deadlifts** are the best for hamstrings and glutes, because they load the hips through a long range of motion.
- **Lunges and split squats**, especially Bulgarian split squats, build quads and glutes and fix side-to-side imbalances.
- **Leg press** lets you train hard with less technical demand and less fatigue in your lower back.

**Useful add-ons**
- **Leg curls** target the hamstrings, especially the short head of the biceps femoris, which hip hinges barely train.
- **Leg extensions** isolate the quads, including the rectus femoris.
- **Calf raises** are needed for calf growth. Squats and deadlifts don't do much for the 

17. 검색 도메인 제한

In [24]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

messages = []
add_user_message(messages, "What's the best exercise for gaining leg muscle?")

# 검색이 길어지면 stop_reason이 "pause_turn"으로 끊길 수 있어 이어서 요청
while True:
    response = chat(messages, tools=[web_search_schema])
    add_assistant_message(messages, response)
    if response.stop_reason != "pause_turn":
        break

print(text_from_message(response))

The barbell **back squat** is usually considered the best single exercise for building leg muscle. It works the quads, glutes, and adductors hard, and to a lesser degree the hamstrings and calves. You can load it heavily and progress it over time, and it's easy to push close to failure.

No single exercise covers everything, though. The best results come from a few movements that hit different muscles:

- **Quads:** Squats (back, front, or hack), leg press, and Bulgarian split squats.
- **Hamstrings:** Romanian deadlifts and leg curls. Seated leg curls are especially effective because they train the hamstrings in a lengthened position.
- **Glutes:** Hip thrusts, deep squats, and lunges or split squats.
- **Calves:** Standing and seated calf raises. Use a full range of motion and a pause at the stretch.

**Principles that matter more than exercise choice:**
1. **Progressive overload.** Add weight or reps over time.
2. **Volume.** Aim for roughly 10–20 hard sets per muscle group per week